In [1]:
import pandas as pd
import sqlite3
import logging

In [2]:
def extract_data(path):
    """ 
    extract_data:This function extract the data
    """
    logging.info("starting the extraction")
    try:
        df=pd.read_csv(path,encoding='windows-1252')
        logging.info("sucesfully extracted data from the superstore data")
        return df
    except Exception as e:
        logging.error(f"File cannot be extracted: {e}")
        return None
        
    

In [3]:
df=extract_data("Superstore.csv")

In [4]:
def transform_the_data(df):
    """
    transforming the data and cleaning it more
    """
    try:
        df.sample(5)
        df.info()
        logging.info("Converting order_date and Shipdate columns datatype to datetime object.")
        df['Order Date']=pd.to_datetime(df['Order Date'])
        df['Ship Date'] = pd.to_datetime(df['Ship Date'])
        logging.info("dropping the duplicate values in the datset")
        initial_count= len(df)
        df = df.drop_duplicates()
        if initial_count > len(df):
            logging.info(f"duplicates found and removed {initial_count - len(df)} duplicate rows.")
        else:
            logging.info("No duplicates rows found.")
            
        logging.info("Creating a new feature and column.")
        df['total_expenditure']=df['Sales']-df['Profit']

        logging.info("filling null values")
        if df['Postal Code'].isnull().any():
            logging.warning("null values founded in column Postal code")
            df['Postal Code']=df['Postal Code'].fillna(0)
        logging.info("transformation done")
        return df

    except Exception as e:
        logging.error("error happended during the transformation")
    
        
        
        
        
    


In [5]:
transform_the_data(df)

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,total_expenditure
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,220.0464
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,512.3580
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,7.7486
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,1340.6085
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,19.8516
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,9990,CA-2014-110422,2014-01-21,2014-01-23,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,South,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.2480,3,0.20,4.1028,21.1452
9990,9991,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,West,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.9600,2,0.00,15.6332,76.3268
9991,9992,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.5760,2,0.20,19.3932,239.1828
9992,9993,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,West,OFF-PA-10004041,Office Supplies,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.6000,4,0.00,13.3200,16.2800


In [8]:
def load_data(df, db_name="superstore.db", table_name="sales_data"):
    """Loads the dataframe into an SQl database."""
    try:
        conn = sqlite3.connect(db_name)
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        logging.info(f"Data successfully loaded into {table_name} table in {db_name}.")
        conn.close()
    except Exception as e:
        logging.error(f"Error loading data to database: {e}")
    

In [10]:
if __name__ == "__main__":
    file_path = r"Superstore.csv"
    
    # Run  the etl Pipeline
    data = extract_data(file_path)
    if data is not None:
        transformed_data = transform_the_data(data)
        if transformed_data is not None:
            load_data(transformed_data)

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

In [12]:
import sqlite3
import pandas as pd

# Connect to the database
conn = sqlite3.connect("superstore.db")

# Query the all rows to verify
query = "SELECT * FROM sales_data"
df_check = pd.read_sql_query(query, conn)
print(df_check)

# Close the connection
conn.close()

      Row ID        Order ID           Order Date            Ship Date  \
0          1  CA-2016-152156  2016-11-08 00:00:00  2016-11-11 00:00:00   
1          2  CA-2016-152156  2016-11-08 00:00:00  2016-11-11 00:00:00   
2          3  CA-2016-138688  2016-06-12 00:00:00  2016-06-16 00:00:00   
3          4  US-2015-108966  2015-10-11 00:00:00  2015-10-18 00:00:00   
4          5  US-2015-108966  2015-10-11 00:00:00  2015-10-18 00:00:00   
...      ...             ...                  ...                  ...   
9989    9990  CA-2014-110422  2014-01-21 00:00:00  2014-01-23 00:00:00   
9990    9991  CA-2017-121258  2017-02-26 00:00:00  2017-03-03 00:00:00   
9991    9992  CA-2017-121258  2017-02-26 00:00:00  2017-03-03 00:00:00   
9992    9993  CA-2017-121258  2017-02-26 00:00:00  2017-03-03 00:00:00   
9993    9994  CA-2017-119914  2017-05-04 00:00:00  2017-05-09 00:00:00   

           Ship Mode Customer ID     Customer Name    Segment        Country  \
0       Second Class    CG-1252